### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "OfficeHome"
MODEL_NAME = "resnet18"

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.set_num_threads(1)


Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1130 09:04:43.380000 23116 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-11-30 09:04:44,808 | INFO | Initialized experiment directories for OfficeHome
2025-11-30 09:04:44,809 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\OfficeHome\logs
2025-11-30 09:04:44,809 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\OfficeHome\checkpoints
2025-11-30 09:04:44,810 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\OfficeHome\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]
CFG["grqo"]["num_tokens"] = 32

{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [5]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=10
)

2025-11-30 07:52:33,531 | INFO | === LODO: Leaving out domain 'Art' ===



=== LODO: Leaving out domain 'Art' ===


2025-11-30 07:55:08,389 | INFO | [Art] Epoch 1/10 | Train - Loss: 2.8734, Cls: 2.8680, GRQO: 0.0067, Acc: 0.4102 | Val - Loss: 2.2854, Cls: 2.2800, GRQO: 0.0068, Acc: 0.4644
2025-11-30 07:55:08,472 | INFO | [Art] New best val acc: 0.4644


[Art] Epoch 1/10 | Train - Loss: 2.8734, Cls: 2.8680, GRQO: 0.0067, Acc: 0.4102 | Val - Loss: 2.2854, Cls: 2.2800, GRQO: 0.0068, Acc: 0.4644
[Art] New best val acc: 0.4644


2025-11-30 07:56:04,776 | INFO | [Art] Epoch 2/10 | Train - Loss: 1.0486, Cls: 1.0478, GRQO: 0.0010, Acc: 0.7979 | Val - Loss: 1.8843, Cls: 1.8742, GRQO: 0.0127, Acc: 0.5303
2025-11-30 07:56:04,908 | INFO | [Art] New best val acc: 0.5303


...

[Real World] Epoch 8/10 | Train - Loss: 0.0126, Cls: 0.0546, GRQO: -0.0524, Acc: 0.9888 | Val - Loss: 1.1274, Cls: 1.1283, GRQO: -0.0012, Acc: 0.7271


2025-11-30 08:38:03,799 | INFO | [Real World] Epoch 9/10 | Train - Loss: -0.0214, Cls: 0.0458, GRQO: -0.0840, Acc: 0.9899 | Val - Loss: 1.1656, Cls: 1.1539, GRQO: 0.0146, Acc: 0.7237


[Real World] Epoch 9/10 | Train - Loss: -0.0214, Cls: 0.0458, GRQO: -0.0840, Acc: 0.9899 | Val - Loss: 1.1656, Cls: 1.1539, GRQO: 0.0146, Acc: 0.7237


2025-11-30 08:39:49,286 | INFO | [Real World] Epoch 10/10 | Train - Loss: -0.0055, Cls: 0.0437, GRQO: -0.0616, Acc: 0.9889 | Val - Loss: 1.1906, Cls: 1.1290, GRQO: 0.0770, Acc: 0.7347
2025-11-30 08:39:49,405 | INFO | [Real World] New best val acc: 0.7347
2025-11-30 08:39:49,406 | INFO | [Real World] Best Acc: 0.7347
2025-11-30 08:39:49,407 | INFO | ------------------------------------------------------------
2025-11-30 08:39:49,411 | INFO | LODO finished | Mean Acc: 0.6176 | Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\OfficeHome\logs\lodo_summary_20251130_083949.json


[Real World] Epoch 10/10 | Train - Loss: -0.0055, Cls: 0.0437, GRQO: -0.0616, Acc: 0.9889 | Val - Loss: 1.1906, Cls: 1.1290, GRQO: 0.0770, Acc: 0.7347
[Real World] New best val acc: 0.7347
[Real World] Best Acc: 0.7347
------------------------------------------------------------
LODO finished | Mean Acc: 0.6176
Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\OfficeHome\logs\lodo_summary_20251130_083949.json


### Baseline

In [ ]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=10
)

2025-11-30 09:04:46,100 | INFO | Initializing ResNet baseline: resnet18


Initializing ResNet baseline: resnet18

=== Baseline LODO: Leaving out domain 'Art' ===


2025-11-30 09:04:46,292 | INFO | === Baseline LODO: Leaving out domain 'Art' ===
2025-11-30 09:05:26,896 | INFO | [Art] Epoch 1/10 | Train - Loss: 2.2335, Acc: 0.5303 | Val Acc: 0.4841


[Art] Epoch 1/10 | Train - Loss: 2.2335, Acc: 0.5303 | Val Acc: 0.4841


2025-11-30 09:06:00,472 | INFO | [Art] Epoch 2/10 | Train - Loss: 0.8469, Acc: 0.8311 | Val Acc: 0.5299


...

[Clipart] Epoch 4/10 | Train - Loss: 0.1987, Acc: 0.9783 | Val Acc: 0.4561


2025-11-30 09:13:18,347 | INFO | [Clipart] Epoch 5/10 | Train - Loss: 0.0957, Acc: 0.9923 | Val Acc: 0.4683


[Clipart] Epoch 5/10 | Train - Loss: 0.0957, Acc: 0.9923 | Val Acc: 0.4683


2025-11-30 09:13:51,401 | INFO | [Clipart] Epoch 6/10 | Train - Loss: 0.0532, Acc: 0.9955 | Val Acc: 0.4658


[Clipart] Epoch 6/10 | Train - Loss: 0.0532, Acc: 0.9955 | Val Acc: 0.4658
